# Load Dataset: BigSalmon2 `data.txt` → Clean CSV

Downloads [BigSalmon2/InformalToFormalDataset](https://github.com/BigSalmon2/InformalToFormalDataset) and extracts clean informal/formal pairs from `data.txt` only.

```
data/
  raw/          ← extracted BigSalmon2 repo
  processed/    ← cleaned CSVs
```

Output: `data/processed/seed_pairs.csv` — 91 rows, columns: `informal`, `formal`

In [ ]:
import requests, zipfile, io, os, glob, re
import pandas as pd
import nltk
import truecase  # pip install truecase

nltk.download('punkt_tab', quiet=True)

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

In [ ]:
ZIP_URL  = 'https://github.com/BigSalmon2/InformalToFormalDataset/archive/refs/heads/main.zip'
DEST_DIR = 'data/raw/BigSalmon2'

print('Downloading repo...', end=' ')
resp = requests.get(ZIP_URL, timeout=60)
resp.raise_for_status()
print(f'{len(resp.content) / 1024:.1f} KB')

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    z.extractall(DEST_DIR)
print('Extracted.')

In [ ]:
matches = glob.glob(f'{DEST_DIR}/**/data.txt', recursive=True)
assert matches, 'data.txt not found'

with open(matches[0], encoding='utf-8') as f:
    lines = f.readlines()

pairs = []
current_informal = None
has_formal = False

for line in lines:
    line = line.strip()
    if not line:
        continue
    if line.lower().startswith('informal english:'):
        current_informal = line[len('informal english:'):].strip()
        has_formal = False
    elif line.lower().startswith('translated into the style of abraham') and not has_formal:
        if current_informal:
            formal = line.split(':', 1)[1].strip()
            if formal:
                pairs.append({'informal': current_informal, 'formal': formal})
                has_formal = True

df = pd.DataFrame(pairs)
print(f'Parsed {len(df)} raw pairs')
df.head(3)

## Preprocessing

The raw data is entirely lowercase with potential whitespace inconsistencies. We apply three fixes:
1. **Whitespace normalization** — collapse tabs/newlines/multiple spaces into a single space
2. **Truecase** — restores proper capitalization for names, cities, etc.
3. **Closing punctuation** — adds a period if the sentence doesn't end with `.`, `!`, or `?`

In [ ]:
def clean(text):
    text = re.sub(r'\s+', ' ', text).strip()       # collapse whitespace
    text = truecase.get_true_case(text)             # restore proper capitalization
    if text and text[-1] not in '.!?':
        text += '.'                                 # ensure closing punctuation
    return text

df['informal'] = df['informal'].apply(clean)
df['formal']   = df['formal'].apply(clean)

df.to_csv('data/processed/seed_pairs.csv', index=False)
print(f'Saved {len(df)} pairs to data/processed/seed_pairs.csv')
df.head(5)